# NLQ with LangChain — Deep Dive

**Goal:** Understand every layer of a Natural Language Query system:
1. Setup — SQLite DB (no Docker needed)
2. `SQLDatabaseChain` — the simple one-shot approach
3. `create_sql_agent` — the iterative, self-correcting approach
4. Why chain vs agent matters — side-by-side comparison
5. Prompt engineering — system prompts, business rules, few-shot examples
6. Memory — conversation history for follow-up questions
7. Putting it all together — a mini REPL that mirrors the production chatbot

**Self-contained:** Uses SQLite (`retail.db` created inline). No PostgreSQL, no Docker.

**LLM used:** Groq (free tier). Set your key in cell 2 or as env var `GROQ_API_KEY`.

## 0. Install dependencies

Run once. If you're using `uv`: `uv pip install langchain langchain-community langchain-groq`

In [ ]:
%pip install -q langchain langchain-community langchain-groq sqlalchemy pydantic

## 1. Configuration — API key

In [ ]:
import os

# Set your Groq API key here, or export GROQ_API_KEY in your shell before launching.
# Get a free key at https://console.groq.com
# DO NOT commit a real key — replace with your own or load from .env
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "your-groq-key-here")
GROQ_MODEL   = os.environ.get("GROQ_MODEL",   "llama-3.3-70b-versatile")

assert GROQ_API_KEY != "your-groq-key-here", "Set GROQ_API_KEY first!"
print(f"Using model: {GROQ_MODEL}")

## 2. Build the SQLite database

Same schema as the production PostgreSQL DB — 4 tables, realistic retail data.
We use `NOW() - INTERVAL` equivalent in SQLite: `datetime('now', '-N days')`.

In [ ]:
import sqlite3
import pathlib

DB_PATH = pathlib.Path("retail.db")

# Remove stale DB so re-running the cell is idempotent
DB_PATH.unlink(missing_ok=True)

conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# --- Schema ---
cur.executescript("""
CREATE TABLE customers (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    name       TEXT    NOT NULL,
    email      TEXT    UNIQUE NOT NULL,
    region     TEXT    NOT NULL,
    created_at TEXT    DEFAULT (datetime('now'))
);

CREATE TABLE products (
    id       INTEGER PRIMARY KEY AUTOINCREMENT,
    name     TEXT    NOT NULL,
    category TEXT    NOT NULL,
    price    REAL    NOT NULL
);

CREATE TABLE orders (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id INTEGER NOT NULL REFERENCES customers(id),
    status      TEXT    NOT NULL,
    ordered_at  TEXT    NOT NULL
);

CREATE TABLE order_items (
    id         INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id   INTEGER NOT NULL REFERENCES orders(id),
    product_id INTEGER NOT NULL REFERENCES products(id),
    quantity   INTEGER NOT NULL,
    unit_price REAL    NOT NULL
);
""")

# --- Customers ---
cur.executemany("INSERT INTO customers (name, email, region) VALUES (?,?,?)", [
    ('Alice Martin', 'alice@example.com', 'North'),
    ('Bob Chen',     'bob@example.com',   'South'),
    ('Carol Davis',  'carol@example.com', 'East'),
    ('Dan Okafor',   'dan@example.com',   'North'),
    ('Eva Singh',    'eva@example.com',   'West'),
    ('Frank Lee',    'frank@example.com', 'South'),  # has cancelled order
    ('Grace Kim',    'grace@example.com', 'East'),
    ('Harry Brown',  'harry@example.com', 'North'),  # no orders
    ('Isla White',   'isla@example.com',  'West'),   # no orders
])

# --- Products ---
cur.executemany("INSERT INTO products (name, category, price) VALUES (?,?,?)", [
    ('Wireless Mouse',      'Electronics', 29.99),
    ('Mechanical Keyboard', 'Electronics', 89.99),
    ('USB-C Hub',           'Electronics', 49.99),
    ('Desk Lamp',           'Office',      34.99),
    ('Notebook (A5)',       'Stationery',   8.99),
    ('Ballpoint Pens x10',  'Stationery',   5.49),
    ('Laptop Stand',        'Electronics', 59.99),
    ('Webcam HD',           'Electronics', 79.99),
])

# --- Orders (recent = within last 30 days) ---
cur.executemany("INSERT INTO orders (customer_id, status, ordered_at) VALUES (?,?,datetime('now',?))", [
    (1, 'delivered', '-5 days'),
    (2, 'shipped',   '-8 days'),
    (3, 'pending',   '-2 days'),
    (4, 'pending',   '-1 day'),
    (5, 'delivered', '-15 days'),
    (6, 'cancelled', '-20 days'),  # Frank's cancelled order — must not appear in sales
    (7, 'pending',   '-3 days'),
    # Older orders (60-75 days ago)
    (1, 'delivered', '-60 days'),
    (2, 'delivered', '-75 days'),
    (3, 'delivered', '-50 days'),
])

# --- Order items ---
cur.executemany("INSERT INTO order_items (order_id, product_id, quantity, unit_price) VALUES (?,?,?,?)", [
    (1,  1, 2, 29.99),  # Alice: 2x Wireless Mouse
    (1,  3, 1, 49.99),  # Alice: 1x USB-C Hub
    (2,  2, 1, 89.99),  # Bob: 1x Mechanical Keyboard
    (2,  7, 1, 59.99),  # Bob: 1x Laptop Stand
    (3,  1, 1, 29.99),  # Carol: 1x Wireless Mouse
    (3,  4, 2, 34.99),  # Carol: 2x Desk Lamp
    (4,  8, 1, 79.99),  # Dan: 1x Webcam HD
    (5,  2, 2, 89.99),  # Eva: 2x Mechanical Keyboard
    (5,  5, 3,  8.99),  # Eva: 3x Notebook
    (6,  6, 5,  5.49),  # Frank: 5x Pens (CANCELLED — should be excluded!)
    (7,  1, 3, 29.99),  # Grace: 3x Wireless Mouse
    (7,  3, 2, 49.99),  # Grace: 2x USB-C Hub
    (8,  2, 1, 89.99),
    (9,  1, 1, 29.99),
    (10, 7, 1, 59.99),
])

conn.commit()
conn.close()
print(f"Created {DB_PATH} successfully")

In [ ]:
# Quick sanity check — verify row counts
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()
for table in ["customers", "products", "orders", "order_items"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table:15s}: {cur.fetchone()[0]} rows")
conn.close()

## 3. LangChain's `SQLDatabase` wrapper

Before touching agents or chains, understand what `SQLDatabase` does.
It is **not** an LLM — it is a Python object that:
- Connects to any SQLAlchemy-compatible DB
- Introspects schema (table names, column types, sample rows)
- Executes SQL strings and returns results as text

Both the chain and the agent use this object under the hood.

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

print("=== Dialect ===")
print(db.dialect)  # 'sqlite' — LangChain injects this into the system prompt

print("\n=== Tables ===")
print(db.get_usable_table_names())

print("\n=== Schema for orders + order_items ===")
print(db.get_table_info(["orders", "order_items"]))

In [ ]:
# SQLDatabase can execute raw SQL and return results as a plain string
result = db.run("SELECT name, region FROM customers LIMIT 3")
print(result)  # plain text — this is what gets fed back to the LLM

**Key insight:** The LLM never touches the database directly.
It generates SQL text → `SQLDatabase.run()` executes it → result text is sent back to the LLM.

## 4. Approach A — `SQLDatabaseChain` (simple, one-shot)

### What it does
```
question → [LLM generates SQL] → [DB executes SQL] → [LLM writes answer]
```
Single pass. The LLM generates SQL once. If it's wrong, there is no retry.

### When to use it
- Simple schemas (1-3 tables)
- Questions that map cleanly to a single SQL query
- Lowest latency (1 LLM call)

### The risk
- LLM hallucinates a column → query fails → chain crashes (or returns an error string)
- No schema introspection by default — you must pass `include_tables` or it will inject the full schema

In [ ]:
from langchain_community.utilities import SQLDatabase
from langchain_community.chains import SQLDatabaseChain
from langchain_groq import ChatGroq
from pydantic import SecretStr

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,       # Deterministic — SQL must be exact, not creative
    api_key=SecretStr(GROQ_API_KEY),
)

# Build the chain — note: no business rules yet, bare minimum
chain = SQLDatabaseChain.from_llm(
    llm=llm,
    db=db,
    verbose=True,        # Prints the SQL it generates — great for learning
    return_intermediate_steps=True,  # Lets us inspect what happened
)

print("Chain built")

In [ ]:
# Run a simple question
result = chain.invoke({"query": "How many customers are in each region?"})

print("\n=== Final Answer ===")
print(result["result"])

print("\n=== Intermediate Steps ===")
for i, step in enumerate(result["intermediate_steps"]):
    print(f"Step {i}: {step}")

In [ ]:
# ⚠️ The chain's blind spot: business rules
# Ask about top products WITHOUT telling it to exclude cancelled orders
result_naive = chain.invoke({"query": "What is the top selling product by quantity?"})
print(result_naive["result"])

# Now manually check: did it include Frank's cancelled order (5x Pens)?
import sqlite3
conn = sqlite3.connect(DB_PATH)
ground_truth = conn.execute("""
    SELECT p.name, SUM(oi.quantity) AS total
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.id
    JOIN products p ON oi.product_id = p.id
    WHERE o.status != 'cancelled'
    GROUP BY p.name ORDER BY total DESC LIMIT 3
""").fetchall()
conn.close()

print("\n=== Correct answer (excluding cancelled) ===")
for row in ground_truth:
    print(row)

The chain likely included the cancelled order unless the LLM happened to guess the business rule.
We'll fix this in Section 6 with a proper system prompt.

## 5. Approach B — `create_sql_agent` (iterative, self-correcting)

### What it does
```
question
  → LLM decides: "I need to see the tables" → calls sql_db_list_tables
  → LLM decides: "I need the schema" → calls sql_db_schema
  → LLM decides: "I'll try this SQL" → calls sql_db_query
  → if error: LLM retries with fixed SQL
  → LLM writes English answer
```

The LLM drives the loop. It **chooses** which tools to call and **when to stop**.

### agent_type options
| Type | How it works | Reliability |
|------|-------------|-------------|
| `zero-shot-react-description` | Outputs `Thought/Action/Observation` text, parsed by regex | Fragile — Llama occasionally mis-formats |
| `tool-calling` | Returns JSON function calls — no text parsing | Robust — native API support |

**Always use `tool-calling` with modern models.**

In [ ]:
from langchain_community.agent_toolkits import create_sql_agent

agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",
    top_k=20,      # Injects LIMIT 20 into generated queries
    verbose=True,  # Prints every tool call — essential for understanding the loop
    agent_executor_kwargs={"handle_parsing_errors": True},
)

print("Agent built")

In [ ]:
# Run the same question as before — watch the verbose output to see the tool calls
result = agent.invoke({"input": "How many customers are in each region?"})
print("\n=== Final Answer ===")
print(result["output"])

In [ ]:
# Deliberate error recovery: ask about a column that doesn't exist
# The agent should detect the error and retry with the correct column name
result = agent.invoke({"input": "What is the average order_value per customer?"})
print("\n=== Final Answer ===")
print(result["output"])
# Notice in verbose output: it may first check schema, then generate correct SQL

## 6. Chain vs Agent — side-by-side comparison

| Dimension | `SQLDatabaseChain` | `create_sql_agent` |
|-----------|-------------------|--------------------|
| LLM calls | 1 (generate + answer) | 3-5 (discover, schema, execute, format) |
| Latency | ~1s | ~3-6s |
| Self-correction | No | Yes |
| Schema discovery | Optional | Always (builds its own context) |
| Multi-step joins | Often fails | Handles reliably |
| Debug visibility | Limited | Full tool-call trace |
| Use when | Simple 1-table queries | Multi-table, unknown schema, production |

In [ ]:
import time

question = "Which customers placed orders in the last 30 days?"

t0 = time.time()
r_chain = chain.invoke({"query": question})
t_chain = time.time() - t0

t0 = time.time()
r_agent = agent.invoke({"input": question})
t_agent = time.time() - t0

print(f"Chain ({t_chain:.1f}s): {r_chain['result'][:200]}")
print()
print(f"Agent ({t_agent:.1f}s): {r_agent['output'][:200]}")

## 7. Prompt Engineering

### 7a. The default system prompt

LangChain injects a system prompt automatically. Let's read it.

In [ ]:
from langchain_community.agent_toolkits.sql.prompt import SQL_PREFIX, SQL_SUFFIX

print("=== SQL_PREFIX (what LangChain injects by default) ===")
print(SQL_PREFIX)
print("\n=== SQL_SUFFIX ===")
print(SQL_SUFFIX)

Notice the `{dialect}` and `{top_k}` placeholders — LangChain fills these from `db.dialect` and `top_k` at runtime.

### 7b. Adding business rules via `prefix`

Business rules go **before** the SQL instructions so they have higher priority.
The pattern: `BUSINESS_RULES + SQL_PREFIX`.

In [ ]:
_BUSINESS_RULES = """You are a helpful data analyst assistant for a retail company.

Business rules — apply these to EVERY query without exception:
- NEVER include orders with status = 'cancelled' when calculating sales, quantities, or revenue.
- "sold", "top products", "best sellers", "revenue" always mean non-cancelled orders only.
  Always add the filter: WHERE orders.status != 'cancelled'
- Valid order statuses are: pending, shipped, delivered, cancelled.
- "last month" means the past 30 days (use datetime('now', '-30 days') for SQLite).
- "yesterday" means the previous calendar day (use date('now', '-1 day')).
- "this week" means the past 7 days (use datetime('now', '-7 days')).
- For sales volume questions: SUM(order_items.quantity) grouped by product.
- For revenue questions: SUM(order_items.quantity * order_items.unit_price).

"""

AGENT_PREFIX = _BUSINESS_RULES + SQL_PREFIX

# Rebuild agent with the custom prefix
agent_with_rules = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",
    top_k=20,
    verbose=True,
    prefix=AGENT_PREFIX,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

print("Agent with business rules built")

In [ ]:
# Now run the cancelled-order test
result = agent_with_rules.invoke({"input": "What is the top selling product by quantity?"})
print("\n=== Answer (should NOT mention Ballpoint Pens or Frank's cancelled order) ===")
print(result["output"])

# Verify: Ballpoint Pens x10 (product_id=6) was in Frank's CANCELLED order
# It should NOT appear in top products if the business rule is working

### 7c. Experimenting with prompt variations

Try these changes and observe the difference in SQL generated.

In [ ]:
# Experiment: weak vs strong business rule phrasing

WEAK_RULES = """
You are a data analyst. Try to exclude cancelled orders when possible.
"""

STRONG_RULES = """
You are a data analyst.
CRITICAL: NEVER include orders with status = 'cancelled' in ANY sales metric.
This is a legal compliance requirement. No exceptions.
Always add: WHERE orders.status != 'cancelled'
"""

def make_agent(rules: str, verbose: bool = True):
    return create_sql_agent(
        llm=llm, db=db,
        agent_type="tool-calling",
        top_k=20,
        verbose=verbose,
        prefix=rules + SQL_PREFIX,
        agent_executor_kwargs={"handle_parsing_errors": True},
    )

# Compare
q = "What is total revenue from all orders?"

print("--- WEAK rules ---")
r_weak = make_agent(WEAK_RULES, verbose=False).invoke({"input": q})
print(r_weak["output"])

print("\n--- STRONG rules ---")
r_strong = make_agent(STRONG_RULES, verbose=False).invoke({"input": q})
print(r_strong["output"])

# Manual ground truth
conn = sqlite3.connect(DB_PATH)
revenue = conn.execute("""
    SELECT SUM(oi.quantity * oi.unit_price)
    FROM order_items oi JOIN orders o ON oi.order_id = o.id
    WHERE o.status != 'cancelled'
""").fetchone()[0]
conn.close()
print(f"\n=== Correct revenue (excl. cancelled): {revenue:.2f} ===")

### 7d. Few-shot examples in the prompt

For tricky date logic, include worked examples directly in the system prompt.
The LLM will pattern-match on them.

In [ ]:
FEW_SHOT_RULES = _BUSINESS_RULES + """
Examples of correct SQL for this database:

Q: Orders placed yesterday?
SQL: SELECT * FROM orders WHERE date(ordered_at) = date('now', '-1 day') AND status != 'cancelled'

Q: Revenue last month?
SQL: SELECT SUM(oi.quantity * oi.unit_price)
     FROM order_items oi JOIN orders o ON oi.order_id = o.id
     WHERE o.ordered_at >= datetime('now', '-30 days')
     AND o.status != 'cancelled'

Q: Top 5 products by units sold this week?
SQL: SELECT p.name, SUM(oi.quantity) AS units
     FROM order_items oi
     JOIN orders o ON oi.order_id = o.id
     JOIN products p ON oi.product_id = p.id
     WHERE o.ordered_at >= datetime('now', '-7 days')
     AND o.status != 'cancelled'
     GROUP BY p.name ORDER BY units DESC LIMIT 5

"""

agent_fewshot = make_agent(FEW_SHOT_RULES, verbose=True)
result = agent_fewshot.invoke({"input": "How much revenue did we make yesterday?"})
print("\n=== Answer ===")
print(result["output"])

## 8. Memory — Conversation History

The SQL agent is **stateless** — each `agent.invoke()` call starts fresh.
Follow-up questions like "What about yesterday?" won't work without memory.

### Two approaches
1. **Plain-text history** (what the production chatbot does) — simple and transparent
2. **LangChain ConversationBufferMemory** — the framework way, more automatic

We'll build both and compare.

### 8a. Approach 1: Plain-text history (manual)

Serialize past turns as a string and prepend to each new question.
Simple. No LangChain abstractions. Easy to debug.

In [ ]:
def build_prompt_with_history(question: str, history: list[dict]) -> str:
    """Prepend last N turns to the current question."""
    if not history:
        return question
    lines = ["Previous conversation:"]
    for turn in history:
        lines.append(f"  User: {turn['question']}")
        lines.append(f"  Assistant: {turn['answer']}")
    lines.append(f"\nCurrent question: {question}")
    return "\n".join(lines)


HISTORY_WINDOW = 5
history: list[dict] = []

def ask(question: str) -> str:
    """Ask a question with history context."""
    prompt = build_prompt_with_history(question, history[-HISTORY_WINDOW:])
    print(f"--- Prompt sent to agent ---\n{prompt}\n---")
    result = agent_with_rules.invoke({"input": prompt})
    answer = result["output"]
    # Store original question, not the enriched prompt, to keep history clean
    history.append({"question": question, "answer": answer})
    return answer

In [ ]:
# Conversation turn 1
a1 = ask("What were the top 3 products sold last month?")
print("\nAnswer 1:", a1)

In [ ]:
# Turn 2: follow-up referring to previous answer
# "them" refers to the products mentioned in turn 1
a2 = ask("Which customers bought them?")
print("\nAnswer 2:", a2)

In [ ]:
# Turn 3: follow-up with relative date — relies on the history context
a3 = ask("What about this week instead of last month?")
print("\nAnswer 3:", a3)

In [ ]:
# Inspect what the history looks like at this point
print("=== Current history ===")
for i, turn in enumerate(history):
    print(f"Turn {i+1}:")
    print(f"  Q: {turn['question']}")
    print(f"  A: {turn['answer'][:150]}...")

### 8b. History window effect

What happens when the window is too small? Try window=1 and ask a question that depends on turn 2.

In [ ]:
# Demonstrate what happens with window=1 (too narrow)
NARROW_WINDOW = 1

narrow_history = [
    {"question": "What were the top 3 products last month?", "answer": "Wireless Mouse, Mechanical Keyboard, USB-C Hub"},
    {"question": "Which customers bought them?", "answer": "Alice, Bob, Carol, Eva, Grace"},
]

# With only 1 turn of context, the agent won't know what "them" referred to in the first turn
prompt_narrow = build_prompt_with_history("What region are they from?", narrow_history[-NARROW_WINDOW:])
prompt_full   = build_prompt_with_history("What region are they from?", narrow_history[-5:])

print("=== Narrow (1 turn) prompt ===")
print(prompt_narrow)
print("\n=== Full (5 turn) prompt ===")
print(prompt_full)

### 8c. Approach 2: LangChain `ConversationBufferMemory`

LangChain's memory objects do the same thing more automatically.
The trade-off: more abstraction, harder to debug, but integrates cleanly with chains.

In [ ]:
from langchain.memory import ConversationBufferWindowMemory

memory = ConversationBufferWindowMemory(
    k=5,                          # Keep last 5 turns (same as HISTORY_WINDOW)
    memory_key="chat_history",    # Variable name injected into the prompt
    return_messages=False,        # Return as plain text (True = return as message objects)
)

# Manually add some turns to inspect the format
memory.save_context(
    {"input":  "What were the top 3 products last month?"},
    {"output": "Wireless Mouse, Mechanical Keyboard, USB-C Hub"}
)
memory.save_context(
    {"input":  "Which customers bought them?"},
    {"output": "Alice, Bob, Carol, Eva, Grace"}
)

# See what LangChain produces
print("=== Memory buffer ===")
print(memory.load_memory_variables({}))

In [ ]:
# Compare: plain text vs LangChain memory format
manual_prompt = build_prompt_with_history(
    "What region are they from?",
    [{"question": "What were the top 3 products last month?", "answer": "Wireless Mouse, Mechanical Keyboard, USB-C Hub"},
     {"question": "Which customers bought them?", "answer": "Alice, Bob, Carol, Eva, Grace"}]
)
lc_memory_text = memory.load_memory_variables({})["chat_history"]

print("=== Manual plain-text approach ===")
print(manual_prompt)

print("\n=== LangChain memory format ===")
print(lc_memory_text)
print("\nCurrent question: What region are they from?")

Both approaches produce nearly identical prompts. The manual approach is more explicit about where the current question begins, which slightly helps the LLM.

### 8d. Memory types overview

LangChain has several memory classes — each with a different retention strategy:

In [ ]:
from langchain.memory import (
    ConversationBufferMemory,        # Keeps ALL turns — grows forever
    ConversationBufferWindowMemory,  # Keeps last k turns — fixed size
    ConversationSummaryMemory,       # Summarises old turns with an LLM — expensive but compact
    ConversationTokenBufferMemory,   # Keeps turns until token budget exhausted
)

# Demonstrate token-based memory — useful when you have a tight context window
token_mem = ConversationTokenBufferMemory(
    llm=llm,
    max_token_limit=200,   # Once this is hit, oldest turns are dropped
    memory_key="chat_history",
)

long_answer = "The top products by revenue were: Mechanical Keyboard ($359.96), USB-C Hub ($199.95), Wireless Mouse ($179.94), Laptop Stand ($179.97), Webcam HD ($79.99). These represent the best-performing electronics in our catalogue."

for i in range(5):
    token_mem.save_context(
        {"input": f"Turn {i} question"},
        {"output": long_answer[:50] + f" (turn {i})"}
    )

print("=== Token-limited memory (only recent turns fit) ===")
print(token_mem.load_memory_variables({}))

## 9. Inspecting the agent internals

How does the agent decide what to do? Let's look at its prompt and tools.

In [ ]:
# The agent's prompt template — the actual text sent to the LLM
print("=== Agent prompt messages ===")
for msg in agent_with_rules.agent.prompt.messages:
    print(f"Type: {type(msg).__name__}")
    if hasattr(msg, 'content'):
        print(f"Content (first 500 chars): {str(msg.content)[:500]}")
    elif hasattr(msg, 'prompt'):
        print(f"Template (first 500 chars): {str(msg.prompt.template)[:500]}")
    print()

In [ ]:
# The agent's available tools
print("=== Agent tools ===")
for tool in agent_with_rules.tools:
    print(f"Name : {tool.name}")
    print(f"Desc : {tool.description[:200]}")
    print()

The four SQL tools the LLM can call:
- `sql_db_list_tables` — discover what tables exist
- `sql_db_schema` — get CREATE TABLE + sample rows
- `sql_db_query` — execute SQL and return results
- `sql_db_query_checker` — ask the LLM to double-check SQL before running it

## 10. Mini REPL — putting it all together

A standalone interactive loop that mirrors how the production chatbot works.
Same pattern as `app.py`, without Chainlit.

In [ ]:
import re

# --- Safety guard (mirrors guard.py) ---
_BLOCKED = re.compile(
    r"^\s*(DROP|DELETE|UPDATE|INSERT|ALTER|TRUNCATE|CREATE|REPLACE|MERGE)\b",
    re.IGNORECASE,
)
_CTE = re.compile(r"^\s*WITH\b.*?AS\s*\(", re.IGNORECASE | re.DOTALL)

def is_safe(sql: str) -> bool:
    clean = re.sub(r"--.*$", "", sql, flags=re.MULTILINE)
    clean = re.sub(r"/\*.*?\*/", "", clean, flags=re.DOTALL)
    if _BLOCKED.match(clean):
        return False
    body = _CTE.sub("", clean).strip()
    return body.upper().startswith("SELECT")

_SQL_KEYWORDS = {"SELECT","WITH","DROP","DELETE","UPDATE","INSERT","ALTER","TRUNCATE","CREATE"}

def looks_like_sql(text: str) -> bool:
    tokens = text.split()
    return bool(tokens) and tokens[0].upper() in _SQL_KEYWORDS


# --- Production-quality agent ---
final_agent = create_sql_agent(
    llm=llm,
    db=db,
    agent_type="tool-calling",
    top_k=20,
    verbose=False,  # Quiet for the REPL — flip to True to see the tool calls
    prefix=AGENT_PREFIX,
    agent_executor_kwargs={"handle_parsing_errors": True},
)

# --- Session state ---
session_history: list[dict] = []


def chat(question: str) -> str:
    """One turn of the chatbot loop."""
    question = question.strip()

    # Guard: only check raw SQL
    if looks_like_sql(question) and not is_safe(question):
        return "Sorry, I can only run read-only queries."

    prompt = build_prompt_with_history(question, session_history[-HISTORY_WINDOW:])
    result = final_agent.invoke({"input": prompt})
    answer = result.get("output", "I couldn't find an answer.")

    session_history.append({"question": question, "answer": answer})
    return answer


print("Mini chatbot ready. Call chat('your question') to interact.")

In [ ]:
# Run a multi-turn conversation
print(chat("What were the top 3 products by units sold?"))

In [ ]:
print(chat("Which customers bought the top product?"))

In [ ]:
print(chat("How many orders are still pending?"))

In [ ]:
# Test the safety guard
print(chat("DROP TABLE orders"))

In [ ]:
# Test cancelled order exclusion
print(chat("What is the total revenue from all orders?"))

conn = sqlite3.connect(DB_PATH)
correct = conn.execute("""
    SELECT ROUND(SUM(oi.quantity * oi.unit_price), 2)
    FROM order_items oi JOIN orders o ON oi.order_id = o.id
    WHERE o.status != 'cancelled'
""").fetchone()[0]
conn.close()
print(f"\nGround truth (excl. cancelled): {correct}")

## 11. Exercises

Try these to deepen your understanding:

**Prompt engineering:**
1. Add a rule: "'active customer' means ordered within the last 90 days". Test it.
2. Add few-shot examples for region-based queries. Does it help accuracy?
3. Remove the business rules entirely — does the cancelled-order answer change?

**Memory:**
4. Change `HISTORY_WINDOW` to 1. Ask a 3-turn conversation. What breaks?
5. Add a fourth memory type: `ConversationSummaryMemory`. What does the summary look like after 5 turns?

**Chain vs Agent:**
6. Try a 3-table join with `SQLDatabaseChain`. Does it succeed? Try the same with the agent.
7. Add `verbose=True` to the agent and count the tool calls for a complex question.

**Schema extension:**
8. Add a `reviews` table (order_id, rating, comment). Ask the agent: "What is the average rating for Electronics?"
9. Try `db = SQLDatabase.from_uri(..., include_tables=["orders", "order_items"])` — the agent can only see 2 tables. What questions break?

**Error handling:**
10. Ask an ambiguous question: "Show me the data". How does the agent respond?
11. Ask about a completely non-existent concept: "What are the subscription tiers?"

## 12. Key mental models

```
SQLDatabase        — Python wrapper: schema introspection + SQL execution
                     The LLM never touches the DB directly

SQLDatabaseChain   — One-shot: question → SQL → execute → answer
                     Fast, simple, brittle on complex queries

create_sql_agent   — Loop: question → discover → schema → SQL → execute → (retry?) → answer
                     Slower, self-correcting, handles multi-table joins reliably

prefix=            — System prompt for the agent
                     Business rules go here, before SQL_PREFIX
                     Filled: {dialect} and {top_k} are substituted by LangChain

agent_type         — "tool-calling" = JSON function calls (reliable)
                     "zero-shot-react-description" = text parsing (fragile)

Memory             — History serialised as text prepended to each prompt
                     The agent itself is stateless — you manage state externally
                     Window size trades context richness vs prompt cost
```